# SpaceX Falcon 9 First Stage Landing Prediction

## Lab 2: Web Scraping Falcon 9 Launch Records from Wikipedia

In this lab, we will perform web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled *List of Falcon 9 and Falcon Heavy launches*. Web scraping is a useful way to collect data when no API is available. We will request the page HTML, parse the launch tables with BeautifulSoup, convert them into a pandas DataFrame, and export the result to `spacex_web_scraped.csv`.

## Objectives

- Send an HTTP GET request to the Wikipedia page and check the response
- Create a BeautifulSoup object and locate the launch records tables
- Extract column names and iterate over rows to build a launch dictionary
- Convert the dictionary into a DataFrame and export it to CSV

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata

### Helper functions

We first define helper functions to clean the raw HTML cells: `date_time` parses the date/time cell, `booster_version` extracts the booster string, `landing_status` reads the booster landing outcome, and `get_mass` extracts the payload mass in kg. The raw wiki text contains reference brackets (e.g. `[5]`) and special characters, so these helpers strip them away.

In [ ]:
def date_time(table_cells):
    """Extract the date and time from a launch table cell."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """Extract the booster version from a launch table cell."""
    out = ''.join([v for i, v in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """Extract the booster landing status from a launch table cell."""
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    """Extract the payload mass (in kg) from a launch table cell."""
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """Extract a cleaned column name from a table header cell."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not (column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name

### Request the Falcon 9 launch Wiki page

We perform an HTTP GET request to fetch the HTML of the Wikipedia page, then create a BeautifulSoup object from the response text. The launch records are stored in HTML tables with the class `wikitable plainrowheaders collapsible`.

In [ ]:
static_url = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
response = requests.get(static_url)
print(response.status_code)

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
soup.title

### Extract all column names from the HTML table header

First we find all `table` elements in the page, then we iterate over the header cells (`th` elements) of the first launch table and apply `extract_column_from_header` to obtain the column names. Some cells return `None` (purely numeric), so we filter those out.

In [ ]:
html_tables = soup.find_all('table')
first_launch_table = html_tables[2]
print(first_launch_table.name)

In [ ]:
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names)

### Create a DataFrame by parsing the launch HTML tables

We create an empty dictionary with the columns we care about, then iterate over every launch table and every row (`tr`) in it. For each row with a flight number, we extract the date/time, booster version, launch site, payload, payload mass, orbit, customer, launch outcome and booster landing status, and append them to the dictionary. Finally we build the DataFrame from the dictionary.

In [ ]:
launch_dict = dict.fromkeys(column_names)

# Remove an irrelevant column
del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

In [ ]:
extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        # Check to see if first table heading is a number corresponding to a launch
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        # Get table element
        row = rows.find_all('td')
        # If it is a number save cells in a dictionary
        if flag:
            extracted_row += 1
            # Flight Number value
            launch_dict['Flight No.'].append(flight_number)
            datatimelist = date_time(row[0])
            # Date value
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            # Time value
            time = datatimelist[1]
            launch_dict['Time'].append(time)
            # Booster version
            bv = booster_version(row[1])
            if not (bv):
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            # Launch Site
            launch_site = row[2].a.string
            launch_dict['Launch site'].append(launch_site)
            # Payload
            payload = row[3].a.string
            launch_dict['Payload'].append(payload)
            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            # Orbit
            orbit = row[5].a.string
            launch_dict['Orbit'].append(orbit)
            # Customer
            if row[6].a is not None:
                customer = row[6].a.string
            else:
                customer = row[6].get_text(strip=True)
            launch_dict['Customer'].append(customer)
            # Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)
            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f"Extracted {extracted_row} launch records.")

In [ ]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head()

### Export the scraped data

The scraped launch records are exported to `spacex_web_scraped.csv`. The complete web scraping flow was: **HTTP GET request to Wikipedia -> create BeautifulSoup object -> locate `wikitable` launch tables -> extract column names from header cells -> iterate rows and parse each cell with helper functions -> build DataFrame -> export CSV**. This dataset can be joined with the API dataset collected in Lab 1.

In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)